In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [ ]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.costs.lse import MLPLSECost
from src.models.gmm_based import GMMEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.train import compute_loss, update_average

In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

In [ ]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [ ]:
from configs.gmm_based.cost import MLPLSECostConfig
from configs.gmm_based.dataset import DatasetConfig, MiniBatchConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [ ]:
# Data
Q_X_UNPAIRED_SAMPLES = 16000 # 1024
R_Y_UNPAIRED_SAMPLES = 16000 # 1024
P_XY_PAIRED_SAMPLES = 16000 # 128

# Optimizer
LR_PAIRED = 3e-4
LR_UNPAIRED = 1e-3

# Sampler
PAIRED_BATCH_SIZE = 128
UNPAIRED_BATCH_SIZE = 128

# Train
MAX_STEPS = 26000
INIT_BY_SAMPLES = True

# Potential
Y_DIM = 2
N_POTENTIALS = 50

# Cost
M_POTENTIALS = 25
LOG_V_M_HIDDEN_CHANNELS = [M_POTENTIALS]
B_M_HIDDEN_CHANNELS = [M_POTENTIALS * Y_DIM]

In [ ]:
dataset_config = DatasetConfig(
    P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
)
minibatch_config = MiniBatchConfig()

cost_config = MLPLSECostConfig(
    m_potentials=M_POTENTIALS,
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
    + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [ ]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [ ]:
X_sampler = StandardNormalSampler(dim=dataset_config.x_dim, device=device)
Y_sampler = SwissRollSampler(dim=dataset_config.y_dim, device=device, dtype=dtype)

In [ ]:
otp_sampler = OTPlanSampler(**minibatch_config.model_dump())

In [ ]:
data_dir = "checkpoints/Tensors"
file_postfix = f"{minibatch_config.cost_function}_{dataset_config.P_XY_paired}"

In [ ]:
X_paired_train, Y_paired_train, X_paired_test, Y_paired_test = generate_paired_data(
    X_sampler, Y_sampler, otp_sampler, dataset_config.P_XY_paired, "./checkpoints/Tensors", file_postfix, device=device
)

In [ ]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, dataset_config.P_XY_paired, device
)

In [ ]:
X_unpaired_test = X_sampler.sample(dataset_config.P_XY_paired)
Y_unpaired_test = Y_sampler.sample(dataset_config.P_XY_paired)

In [ ]:
if dataset_config.Q_X_unpaired > 0:
    source_data = X_sampler.sample(dataset_config.Q_X_unpaired)
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if dataset_config.R_Y_unpaired > 0:
    target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [ ]:
cost = MLPLSECost(**cost_config.model_dump())

In [ ]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [ ]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

## 5. Optimizers initialization

In [ ]:
unpaired_params_to_update = [model._log_w_n, model._a_n, model._log_A_n]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [ ]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [ ]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_Swiss_Roll_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"MINIBATCH_COST_{minibatch_config.cost_function}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [ ]:
starting_points = torch.tensor([[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]])
num_ending_points = 64

In [ ]:
num_starting_points_paired = 5
indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [ ]:
gt_Y_points = get_GT_points(X_sampler, Y_sampler, otp_sampler, starting_points)

In [ ]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired loss": D_loss_unpaired.item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)
    
    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    wandb.log(
        {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
        step=step,
    )

    wandb.log({r"$-f^c(x)$": -output_unpaired["f_c"].mean().item()}, step=step)
    wandb.log({r"$-f(y)$": -output_unpaired["f"].mean().item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(output_unpaired["A_n"])}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(output_unpaired["A_n"])}, step=step)

    if step % train_config.plot_every == 0:
        A_dict = plot_A_parameters(model, log=True)
        B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        if num_starting_points_paired > 0:
            Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
        else:
            Z_dict = plot_Z_parameters(model, starting_points, log=True)
        distr_dict = plot_swiss_roll(
            {f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_UNPAIRED_SAMPLES}, R={R_Y_UNPAIRED_SAMPLES}": model},
            X_sampler,
            Y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            log=True,
        )
        wandb.log(A_dict | B_dict | Z_dict | distr_dict)

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
wandb: ERROR Dropped streaming file chunk (see wand

## 7. Naive Training

In [ ]:
LR_NAIVE = 1e-3

In [ ]:
cost = MLPLSECost(**cost_config.model_dump())

In [ ]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [ ]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

In [ ]:
D_opt_naive = torch.optim.Adam(model.parameters(), lr=LR_NAIVE)

In [ ]:
if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_naive.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    log_w_n = model.log_w_n()
    a_n = model.a_n()
    A_n = model.A_n()

    cond_distr_unpaired = model.get_conditional_distribution(
        X.repeat(train_config.unpaired_batch_size, 1), log_w_n, a_n, A_n
    )
    fwd = cond_distr_unpaired.log_prob(Y.repeat(train_config.unpaired_batch_size, 1))
    D_loss_unpaired = -torch.log(
        torch.mean(torch.exp(fwd.reshape(train_config.unpaired_batch_size, train_config.unpaired_batch_size)), dim=-1)
    ).mean()

    wandb.log({f"Unpaired loss": D_loss_unpaired.item()}, step=step)

    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    cond_distr_paired = model.get_conditional_distribution(X_paired, log_w_n, a_n, A_n)
    D_loss_paired = -cond_distr_paired.log_prob(Y_paired).mean()

    wandb.log({f"Paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_naive.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    wandb.log(
        {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
        step=step,
    )

    wandb.log({f"lam_min(A_n)": torch.min(A_n)}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(A_n)}, step=step)

    if step % train_config.plot_every == 0:
        A_dict = plot_A_parameters(model, log=True)
        B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        if num_starting_points_paired > 0:
            Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
        else:
            Z_dict = plot_Z_parameters(model, starting_points, log=True)
        distr_dict = plot_swiss_roll(
            {f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_UNPAIRED_SAMPLES}, R={R_Y_UNPAIRED_SAMPLES}": model},
            X_sampler,
            Y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            log=True,
        )
        wandb.log(A_dict | B_dict | Z_dict | distr_dict)

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

## Plotting

In [ ]:
Q_X_unpaired_samples_list = [0, 1024]
R_Y_unpaired_samples_list = [0, 1024]
log_step = 99000

In [ ]:
models_dict = dict()

for i, Q_X_unpaired_samples in enumerate(Q_X_unpaired_samples_list):
    for j, R_Y_unpaired_samples in enumerate(R_Y_unpaired_samples_list):
        model = GMMEOT(
            y_dim=Y_DIM,
            n_potentials=N_POTENTIALS,
            cost=cost,
        ).to(dtype)
        exp_name = EXP_NAME.replace(
            f"Q_X_UNPAIRED_{Q_X_UNPAIRED_SAMPLES}_R_Y_UNPAIRED_{R_Y_UNPAIRED_SAMPLES}_",
            f"Q_X_UNPAIRED_{Q_X_unpaired_samples}_R_Y_UNPAIRED_{R_Y_unpaired_samples}_",
        )
        print(exp_name)
        output_path = "../checkpoints/{}".format(exp_name)
        model.load_state_dict(torch.load(os.path.join(output_path, f"D_{log_step}.pt"), map_location=device))
        title = f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_unpaired_samples}, R={R_Y_unpaired_samples}"
        models_dict[title] = model

In [ ]:
plot_swiss_roll(
    models_dict,
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 